Cadena Compleja (Simulación de RAG)
En la sección "More complex chain", tu cuaderno intenta hacer una búsqueda en una pequeña base de datos para darle "contexto" a la IA antes de responder (esto se conoce como RAG).

Este patrón se conoce como RAG (Generación Aumentada por Recuperación) y sirve para que la Inteligencia Artificial responda preguntas usando datos tuyos que no venían en su entrenamiento original.

In [1]:
from dotenv import load_dotenv

load_dotenv: El "guardia de seguridad". Se encarga de buscar tu archivo oculto .env para leer tu API Key de forma segura sin exponerla en el código.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

ChatPromptTemplate: El "molde". Sirve para crear plantillas de texto con espacios en blanco (como {contexto}) que se rellenarán dinámicamente después.

ChatGoogleGenerativeAI: El "cerebro conversacional" (en tu caso, Gemini). Es el modelo que lee y redacta las respuestas.

GoogleGenerativeAIEmbeddings: El "traductor matemático". La IA no entiende palabras como los humanos; necesita convertir el texto en una lista de números (vectores) para poder medir qué tan parecidas son dos frases entre sí.

InMemoryVectorStore: La "caja de zapatos inteligente". Una base de datos súper simple que guarda esos números en la memoria RAM de tu computadora mientras el programa esté corriendo.

StrOutputParser: El "filtro de limpieza". Los modelos de lenguaje devuelven respuestas con metadatos complejos (metatags, tokens, ID de respuesta). Este componente limpia todo eso y te entrega solo el texto puro.

RunnablePassthrough: La "cinta transportadora". Deja pasar la pregunta del usuario tal cual llegó, sin modificarle ni una sola letra.

In [3]:
load_dotenv()

True

¿Qué hace? Activa al guardia de seguridad. Lee tu archivo .env y pone tu clave en la memoria del sistema para que Gemini sepa que tienes permiso de usar sus servicios.

In [9]:
# 1. CONFIGURAR EMBEDDINGS (Convertir texto a números/vectores)
# Usamos el modelo oficial de Google para embeddings de texto
# PASO 1 CORREGIDO: Asegúrate de que esta línea esté así
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

¿Qué hace? Configuramos nuestro modelo traductor. Cada vez que le demos una frase a este objeto embeddings, lo enviará a Google y nos devolverá su mapa matemático (vector). (Nota: Si en tu cuenta este nombre exacto da error, recuerda cambiarlo por el que te funcione en tu región).

In [10]:
# 2. CREAR BASE DE DATOS EN MEMORIA
vectorstore = InMemoryVectorStore.from_texts(
    ["Harrison trabajó en Kensho", "A los osos les gusta comer miel"],
    embedding=embeddings
)

InMemoryVectorStore.from_texts(...): Aquí creamos nuestra base de datos. Le pasamos dos textos de ejemplo y le damos nuestro traductor (embedding=embeddings). Automáticamente, el sistema traduce ambas frases a números y las guarda en la RAM.

In [11]:
# El retriever es el "buscador" dentro de nuestra base de datos
retriever = vectorstore.as_retriever()


vectorstore.as_retriever(): Convertimos esa base de datos en un retriever (Buscador). Su único trabajo en el mundo es: "Dame una pregunta y yo revolveré la caja de zapatos para devolverte la frase que más se relacione matemáticamente".

In [12]:
# 3. PLANTILLA DE PROMPT
template = """Responde la pregunta basándote únicamente en el siguiente contexto:
{context}

Pregunta: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

template: Diseñamos las instrucciones estrictas. Nota las llaves {context} y {question}. Son "campos vacíos" que nuestra fábrica rellenará de forma automática más adelante.

model: Encendemos los motores de Gemini 2.5 Flash Lite, nuestra IA elegida para procesar y razonar.

In [13]:
# Usamos el modelo que ya comprobaste que te funciona perfectamente
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [14]:
# 4. LA CADENA COMPLEJA CON LCEL (RAG)
chain = (
    {
        "context": retriever, 
        "question": RunnablePassthrough()
    }
    | prompt 
    | model 
    | StrOutputParser()
)

¿Cómo fluyen los datos por aquí adentro? Imagina que ejecutas el programa enviando la pregunta: "¿Dónde trabajó Harrison?".

El Diccionario Inicial recibe la pregunta y se divide en dos tareas en paralelo:

Enviará la pregunta al retriever. El buscador va a la base de datos, encuentra que "Harrison trabajó en Kensho" es la respuesta más parecida y la guarda dentro de la clave "context".

Al mismo tiempo, RunnablePassthrough() toma la pregunta original ("¿Dónde trabajó Harrison?") sin tocarle nada y la guarda dentro de la clave "question".

El tubo | agarra ese diccionario con los dos datos listos y se los lanza al prompt.

El prompt toma lo que hay en "context" y "question", lo inyecta en las llaves del molde y genera este texto final unificado:

"Responde la pregunta basándote únicamente en el siguiente contexto: Harrison trabajó en Kensho. Pregunta: ¿Dónde trabajó Harrison?"

El siguiente tubo | toma ese bloque de texto y se lo entrega directamente a model (Gemini).

Gemini lee las instrucciones, procesa la información y genera una respuesta cruda estructurada.

El último tubo | le pasa esa respuesta a StrOutputParser(), que remueve la basura técnica y te deja solo el string limpio.

In [15]:
# 5. EJECUCIÓN
print(chain.invoke("¿Dónde trabajó Harrison?"))

Harrison trabajó en Kensho.


¿Qué hace? El método .invoke() es el que presiona el botón de "PLAY" en toda la fábrica que acabamos de armar, pasándole el dato inicial. El resultado final viajará por toda la cadena y el comando print lo mostrará en tu pantalla limpia y llanamente:

Harrison trabajó en Kensho.